# 19 - Generasi Prediksi RAG (K1-K5)

Lanjutan notebook 18. Install Ollama di Kaggle, registrasikan 4 GGUF, jalankan pipeline hybrid search + rerank + generate untuk K1-K4 (Ollama) dan K5 (DeepSeek API).

## 1. Setup paket

In [1]:
import subprocess, sys
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "pandas==2.2.3", "requests>=2.32,<3", "huggingface-hub==0.27.1",
        "langchain==0.3.14", "langchain-core==0.3.29", "langchain-community==0.3.14",
        "sentence-transformers==3.3.1", "einops>=0.7,<1",
        "faiss-cpu==1.9.0.post1", "rank-bm25==0.2.2"])
    print("Selesai. RESTART KERNEL, lalu set INSTALL_DEPENDENCIES=False sebelum lanjut.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.29 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.29 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.27.1 which is incompatible.
gradio 5.50

Selesai. RESTART KERNEL, lalu set INSTALL_DEPENDENCIES=False sebelum lanjut.


## 2. Muat konfigurasi dari notebook 18

`RUN_NAME` harus sama dengan notebook 18. Kalau sesi Kaggle baru (bukan lanjutan sesi 18), attach folder hasil 18 sebagai Kaggle input dan isi `PRIOR_RUN_DIR`.

In [2]:
from pathlib import Path
import os, json, hashlib, time, platform, random
import importlib.metadata as pkg_metadata
import pandas as pd
import numpy as np
import requests

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

def read_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None
    if required and not value:
        raise RuntimeError(f"Aktifkan secret {name} pada Kaggle Add-ons > Secrets.")
    return value

RUN_NAME = "Lengkap"
PRIOR_RUN_DIR = "/kaggle/input/notebooks/muhammadnabilul/19-generate-prediction-rag-k1-k5/rag_results/pilot_v2"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "rag_results" / RUN_NAME
KB_DIR = WORK / "rag_assets" / "kb"

if PRIOR_RUN_DIR and not (OUT / "manifest.json").exists():
    import shutil
    OUT.mkdir(parents=True, exist_ok=True)
    for p in Path(PRIOR_RUN_DIR).rglob("*"):
        if p.is_file():
            target = OUT / p.relative_to(PRIOR_RUN_DIR)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, target)

manifest_18 = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
MODEL_REGISTRY = manifest_18["model_registry"]
SCENARIOS = manifest_18["scenarios"]
ASSET_REVISIONS = manifest_18["asset_revisions"]
DATA_SPLIT = manifest_18["data_split"]
SEED = manifest_18["seed"]

def _ensure_kb():
    from huggingface_hub import hf_hub_download
    token = read_secret("HF_Faiss")
    for name in ["chunk_metadata.jsonl", "faiss_index.bin"]:
        if not (KB_DIR / name).exists():
            hf_hub_download("Makaareeem/publikasi-rag-knowledge-base", name, repo_type="dataset",
                             revision=ASSET_REVISIONS["kb"], token=token, local_dir=KB_DIR)
_ensure_kb()
print("Dimuat:", RUN_NAME, "-", len(SCENARIOS), "skenario -",
      len(list((OUT / "predictions").glob("*.json"))), "prediksi lama ter-restore")

chunk_metadata.jsonl:   0%|          | 0.00/45.7M [00:00<?, ?B/s]

faiss_index.bin:   0%|          | 0.00/69.4M [00:00<?, ?B/s]

Dimuat: Lengkap - 9 skenario - 1246 prediksi lama ter-restore


In [3]:
import json
from pathlib import Path

deleted = 0
for p in (OUT / "predictions").glob("*deepseek-flash*.json"):
    row = json.loads(p.read_text(encoding="utf-8"))
    if not row.get("answer", "").strip():
        p.unlink()
        deleted += 1
print(deleted, "file kosong dihapus, akan digenerate ulang saat loop utama dijalankan lagi")

130 file kosong dihapus, akan digenerate ulang saat loop utama dijalankan lagi


## 3. Registrasi ulang konstanta (harus identik notebook 18)

In [4]:
HF_TOKEN = read_secret("HF_Faiss")
HF_KB_REPO = "Makaareeem/publikasi-rag-knowledge-base"
EMBEDDING_MODEL_ID = "nomic-ai/nomic-embed-text-v1.5"
QUERY_PREFIX = "search_query: "
RETRIEVAL_K = 5
TOP_K_DENSE = 20
TOP_K_SPARSE = 20
TOP_K_RERANK_CANDIDATES = 20
RRF_K = 60
MAX_NEW_TOKENS = 512
RERANKER_MODEL_ID = "madebyaris/rerank-indonesia"
RERANK_SCORE_THRESHOLD = -1.0
INFERENCE_URL = "http://127.0.0.1:11434"
NUM_CTX = 8192
PROTOCOLS = ["end_to_end", "controlled"]
REFERENCE_MODEL = "llama3.2-base"
TOP_K = 5
 
SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS "
    "berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum "
    "menjawab, merujuk sumber HANYA dengan nomor sitasi seperti [1] [2], jangan "
    "menyebut ulang judul dokumen dalam kalimat jawaban karena berisiko salah "
    "mengaitkan dengan sumber lain. HANYA gunakan angka atau fakta yang secara "
    "eksplisit tertulis dalam konteks yang diberikan. JANGAN menambahkan angka dari "
    "pengetahuan Anda sendiri walau terasa familiar atau masuk akal. Jika konteks "
    "memuat TARGET, RENCANA, atau PROYEKSI, jelaskan dengan jelas bahwa itu adalah "
    "target/rencana, BUKAN angka realisasi. Jika konteks tidak memuat informasi yang "
    "relevan, katakan dengan jujur bahwa Anda tidak memiliki informasi tersebut."
)
assert digest(SYSTEM_PROMPT) == manifest_18["system_prompt_hash"], "SYSTEM_PROMPT tidak sama dengan notebook 18."
 
QUERY_EXPANSION_TEMPERATURE = 0.3
QUERY_EXPANSION_MAX_TOKENS = 80
QUERY_EXPANSION_SYSTEM_PROMPT = (
    'Tugas Anda: tulis ulang SATU pertanyaan menjadi SATU kalimat pencarian yang lebih '
    'deskriptif untuk sistem pencarian dokumen statistik BPS. ATURAN KETAT: 1. Output '
    'HANYA kalimat hasil tulis ulang, tanpa basa-basi/kutip/penjelasan. 2. JANGAN '
    'menjawab pertanyaannya atau menyebut angka spesifik. 3. JANGAN berkomentar '
    'tentang teks/dokumen. 4. JANGAN basa-basi asisten. 5. Maksimal 1 kalimat.'
)

# K5 - generator API terpisah dari judge RAGAS (self-preference bias, Zheng et al. 2023)
COMMERCIAL_MODEL = {"provider": "deepseek", "model": "deepseek-flash",
                     "base_url": "https://api.deepseek.com", "api_key_secret": "API_Deepseek"}

## 4. Muat test set dan gold chunk dari notebook 18

In [5]:
test = pd.read_csv(OUT / "test_selected.csv", dtype=str, keep_default_na=False)
gold = pd.read_json(OUT / "gold_chunks.jsonl", lines=True)
print(len(test), "pertanyaan,", len(gold), "gold chunk records")

89 pertanyaan, 89 gold chunk records


## 5. Pasang Ollama dan registrasikan 4 GGUF

In [6]:
import shutil as _shutil

# hash berkas besar bertahap, tanpa load penuh ke RAM
def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# install Ollama di sesi Kaggle (idempotent) dan pastikan server hidup
def ensure_kaggle_ollama():
    if not _shutil.which("ollama"):
        subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
        installer = WORK / "install_ollama.sh"
        resp = requests.get("https://ollama.com/install.sh", timeout=60)
        resp.raise_for_status()
        installer.write_text(resp.text, encoding="utf-8")
        subprocess.run(["sh", str(installer)], check=True)
    try:
        requests.get(INFERENCE_URL + "/api/version", timeout=5).raise_for_status()
        return
    except requests.RequestException:
        pass
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MAX_LOADED_MODELS="1")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            requests.get(INFERENCE_URL + "/api/version", timeout=2).raise_for_status()
            return
        except requests.RequestException:
            time.sleep(1)
    raise RuntimeError("Ollama belum siap; periksa log sesi Kaggle.")

# unduh 4 GGUF dari HF, buat Modelfile, daftarkan ke Ollama via `ollama create`
def prepare_models():
    from huggingface_hub import hf_hub_download
    info = {}
    for key, model in MODEL_REGISTRY.items():
        path = Path(hf_hub_download(model["gguf_repo"], model["gguf_filename"],
                             revision=ASSET_REVISIONS[key], token=HF_TOKEN))
        lines = [f'FROM "{path.as_posix()}"']
        if model["system_role"]:
            lines.append('SYSTEM  + SYSTEM_PROMPT + ')
        lines += ["PARAMETER temperature 0", f"PARAMETER num_predict {MAX_NEW_TOKENS}",
                  f"PARAMETER repeat_penalty {model['repeat_penalty']}", f"PARAMETER num_ctx {NUM_CTX}"]
        modelfile = WORK / "rag_assets" / f"{key}.Modelfile"
        modelfile.parent.mkdir(parents=True, exist_ok=True)
        modelfile.write_text("\n".join(lines), encoding="utf-8")
        subprocess.run(["ollama", "create", model["ollama_name"], "-f", str(modelfile)], check=True)
        info[key] = {"repo": model["gguf_repo"], "filename": path.name,
                     "revision": ASSET_REVISIONS[key], "sha256": file_hash(path)}
    return info

ensure_kaggle_ollama()
MODEL_ARTIFACTS = prepare_models()
print("Model terdaftar:", list(MODEL_ARTIFACTS))

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 124 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,054 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Llama-3.2-3B-Instruct.Q4_K_M.gguf:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

gathering model components ⠋ gathering model components ⠙ gathering model components ⠸ gathering model components ⠸ gathering model components ⠼ gathering model components ⠦ gathering model components ⠧ gathering model components ⠧ gathering model components ⠏ gathering model components ⠋ gathering model components ⠋ gathering model components ⠙ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠙ gathering model components ⠙ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠼ gathering mode

Llama-3.2-3B-Instruct.Q4_K_M.gguf:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

...

gemma-2-2b-it.Q4_K_M.gguf:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠇ gathering model components ⠋ gathering model components ⠋ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠧ gathering model components ⠏ gathering model components ⠋ gathering model components ⠋ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠼ gathering mode

gemma-2-2b-it.Q4_K_M.gguf:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠼ gathering model components ⠦ gathering model components ⠦ gathering model components ⠧ gathering model components ⠏ gathering model components ⠏ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠦ gathering model components ⠇ gathering model components ⠏ gathering model components ⠏ gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠼ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠏ gathering model components ⠏ gathering model components ⠙ gathering model components ⠙ gathering model components ⠹ gathering model components ⠼ gathering mode

Model terdaftar: ['llama3.2-base', 'llama3.2-finetuned', 'gemma2-base', 'gemma2-finetuned']


## 6. Fungsi retrieval (dense + BM25 + RRF + rerank)

In [7]:
import faiss
from huggingface_hub import hf_hub_download
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer, CrossEncoder

class NomicQueryEmbeddings(Embeddings):
    def __init__(self, model_id=EMBEDDING_MODEL_ID):
        self.model = SentenceTransformer(model_id, trust_remote_code=True,
                                          revision=ASSET_REVISIONS["embedding"], device="cpu")

    def embed_query(self, text):
        return self.model.encode(QUERY_PREFIX + text, normalize_embeddings=True).tolist()

    def embed_documents(self, texts):
        raise NotImplementedError("Dokumen sudah diembed pada NB07.")

# baca chunk_metadata.jsonl jadi list Document LangChain
def load_documents(metadata_path):
    documents = []
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            documents.append(Document(page_content=record["text"], metadata=record))
    return documents

# bungkus index FAISS mentah + dokumen jadi vectorstore LangChain
def adapt_faiss_to_langchain(index_path, documents, embeddings):
    raw_index = faiss.read_index(str(index_path))
    docstore_dict, index_to_docstore_id = {}, {}
    for doc in documents:
        doc_id = str(doc.metadata["faiss_id"])
        docstore_dict[doc_id] = doc
        index_to_docstore_id[doc.metadata["faiss_id"]] = doc_id
    return FAISS(embedding_function=embeddings, index=raw_index,
                 docstore=InMemoryDocstore(docstore_dict), index_to_docstore_id=index_to_docstore_id)

# muat vectorstore FAISS + BM25 dari artefak KB yang sudah diunduh notebook 18
def build_indices():
    documents = load_documents(KB_DIR / "chunk_metadata.jsonl")
    embeddings = NomicQueryEmbeddings()
    vectorstore = adapt_faiss_to_langchain(KB_DIR / "faiss_index.bin", documents, embeddings)
    bm25 = BM25Retriever.from_documents(documents, preprocess_func=lambda t: t.lower().split())
    bm25.k = TOP_K_SPARSE
    return vectorstore, bm25

# gabung ranking dense+sparse pakai Reciprocal Rank Fusion
def reciprocal_rank_fusion(rankings, k=RRF_K):
    scores, lookup = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            doc_id = doc.metadata["faiss_id"]
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            lookup[doc_id] = doc
    return [lookup[i] for i in sorted(scores, key=scores.get, reverse=True)]

# ambil top-k dense + top-k sparse untuk query asli + hasil ekspansi, gabung RRF
def hybrid_search(vectorstore, bm25, queries):
    rankings = []
    for q in queries:
        rankings.append([d for d, _ in vectorstore.similarity_search_with_score(q, k=TOP_K_DENSE)])
        rankings.append(bm25.invoke(q)[:TOP_K_SPARSE])
    return reciprocal_rank_fusion(rankings)[:TOP_K_RERANK_CANDIDATES]

_reranker = None

def get_reranker():
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder(RERANKER_MODEL_ID, revision=ASSET_REVISIONS["reranker"], device="cpu")
    return _reranker

# nilai ulang kandidat dengan cross-encoder, kembalikan (lolos threshold, seluruh ranking)
def rerank(question, docs, top_k=RETRIEVAL_K):
    if not docs:
        return [], []
    model = get_reranker()
    scores = model.predict([[question, d.page_content] for d in docs])
    ranked = sorted(zip(docs, (float(s) for s in scores)), key=lambda p: p[1], reverse=True)
    passed = [(d, s) for d, s in ranked if s >= RERANK_SCORE_THRESHOLD]
    return passed[:top_k], ranked

# identitas evaluasi unik (chunk_id bisa berulang di KB dengan faiss_id beda)
def evaluation_chunk_id(meta):
    return meta["chunk_id"] + "::faiss=" + str(meta["faiss_id"])

## 7. Fungsi generate (dispatcher Ollama vs API komersial)

In [8]:
import re

_PREAMBLE_RE = re.compile(
    r'^(tentu(\s+saja)?|baik|oke|ok|teks (ini |tersebut )?menyatakan|berdasarkan '
    r'(teks|dokumen)|dokumen (ini |tersebut )?menyebutkan|saya (dapat|bisa) membantu)'
    r'[^\n.:]*[.:,]\s*', re.IGNORECASE)

def _clean_expansion(text):
    cleaned = text.strip().strip('"\'“”')
    prev = None
    while prev != cleaned:
        prev = cleaned
        cleaned = _PREAMBLE_RE.sub("", cleaned).strip().strip('"\'“”')
    return cleaned

def parse_json_response(raw, caller):
    text = raw.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"[{caller}] Gagal parse JSON:\n{raw}\n")
        return None

# panggil Ollama /api/chat - dipakai untuk K1-K4
def generate_ollama(model_name, messages, repeat_penalty=1.0, temperature=0,
                     num_predict=MAX_NEW_TOKENS, response_format=None):
    payload = {"model": model_name, "messages": messages, "stream": False,
               "options": {"temperature": temperature, "num_predict": num_predict,
                            "repeat_penalty": repeat_penalty, "seed": SEED, "num_ctx": NUM_CTX}}
    if response_format:
        payload["format"] = response_format
    r = requests.post(f"{INFERENCE_URL}/api/chat", json=payload, timeout=(10, 900))
    r.raise_for_status()
    return r.json()["message"]["content"]

# panggil API OpenAI-compatible (DeepSeek untuk K5) - format request identik ke OpenAI SDK
def generate_api(provider_cfg, messages, temperature=0, num_predict=MAX_NEW_TOKENS, response_format=None):
    key = read_secret(provider_cfg["api_key_secret"], required=True)
    payload = {"model": provider_cfg["model"], "messages": messages,
               "temperature": temperature, "max_tokens": num_predict,
               "reasoning_effort": "none"}
    if response_format:
        payload["response_format"] = {"type": "json_object"}
    r = requests.post(f"{provider_cfg['base_url']}/chat/completions", json=payload,
                       headers={"Authorization": f"Bearer {key}"}, timeout=(10, 300))
    r.raise_for_status()
    data = r.json()["choices"][0]
    if not data["message"]["content"]:
        raise RuntimeError(f"Content kosong, finish_reason={data.get('finish_reason')}")
    return data["message"]["content"]
 
# dispatcher: model_key ada di MODEL_REGISTRY -> Ollama, kalau tidak -> COMMERCIAL_MODEL
def generate_answer(model_key, messages, repeat_penalty=1.0, temperature=0,
                     num_predict=MAX_NEW_TOKENS, response_format=None):
    if model_key in MODEL_REGISTRY:
        return generate_ollama(MODEL_REGISTRY[model_key]["ollama_name"], messages,
                                repeat_penalty, temperature, num_predict, response_format)
    assert model_key == COMMERCIAL_MODEL["model"], f"Model tidak dikenal: {model_key}"
    return generate_api(COMMERCIAL_MODEL, messages, temperature, num_predict, response_format)
 
def health():
    try:
        requests.get(INFERENCE_URL + "/api/tags", timeout=5).raise_for_status()
        return True
    except requests.RequestException:
        return False

## 8. Ekspansi kueri, sitasi, dan konstruksi prompt

In [9]:
# model_cfg generik: dict dari MODEL_REGISTRY (system_role, repeat_penalty) ATAU COMMERCIAL_MODEL
def _system_role(model_key):
    if model_key in MODEL_REGISTRY:
        return MODEL_REGISTRY[model_key]["system_role"], MODEL_REGISTRY[model_key]["repeat_penalty"]
    return True, 1.0  # API commercial: anggap dukung system role native
 
def expand_query(question, model_key):
    system_role, repeat_penalty = _system_role(model_key)
    if system_role:
        messages = [{"role": "system", "content": QUERY_EXPANSION_SYSTEM_PROMPT},
                    {"role": "user", "content": question}]
    else:
        messages = [{"role": "user", "content": f"{QUERY_EXPANSION_SYSTEM_PROMPT}\n\n{question}"}]
    raw = generate_answer(model_key, messages, repeat_penalty, QUERY_EXPANSION_TEMPERATURE,
                           QUERY_EXPANSION_MAX_TOKENS)
    expanded = _clean_expansion(raw)
    return [expanded] if expanded else []
 
def format_context(docs):
    blocks = []
    for i, d in enumerate(docs, 1):
        header = f"[{i}] {d.metadata.get('document_title', '-')}"
        if d.metadata.get("section_id"):
            header += f" | {d.metadata['section_id']}"
        blocks.append(f"{header}\n{d.page_content}")
    return "\n\n".join(blocks)
 
def build_user_content(question, docs, use_rag):
    if use_rag and not docs:
        return f"Pertanyaan: {question}\n\n(Tidak ditemukan bagian dokumen yang cukup relevan.)"
    if not docs:
        return f"Pertanyaan: {question}"
    return f"Konteks:\n{format_context(docs)}\n\nPertanyaan: {question}"
 
def build_messages(question, docs, system_role, use_rag=True):
    user_content = build_user_content(question, docs, use_rag)
    if system_role:
        return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_content}]
    return [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_content}"}]

## 9. Bekukan manifest generasi (perluasan manifest notebook 18)

In [10]:
def package_versions():
    names = ["pandas", "numpy", "requests", "sentence-transformers",
             "langchain-core", "langchain-community", "faiss-cpu", "rank-bm25"]
    return {n: pkg_metadata.version(n) for n in names}
 
def freeze_manifest():
    runtime = requests.get(INFERENCE_URL + "/api/version", timeout=15)
    runtime.raise_for_status()
    gen_manifest = {**manifest_18, "protocols": PROTOCOLS, "reference_model": REFERENCE_MODEL,
        "top_k": TOP_K, "models": MODEL_ARTIFACTS, "num_ctx": NUM_CTX,
        "commercial_model": COMMERCIAL_MODEL,
        "prompts_hash": digest([SYSTEM_PROMPT, QUERY_EXPANSION_SYSTEM_PROMPT]),
        "kb_hashes": {n: file_hash(KB_DIR / n) for n in ["chunk_metadata.jsonl", "faiss_index.bin"]},
        "versions": package_versions(), "ollama": runtime.json()}
    gen_manifest["experiment_id"] = digest(gen_manifest)
    path = OUT / "manifest.json"
    existing = json.loads(path.read_text(encoding="utf-8"))
    if "experiment_id" in existing and existing["experiment_id"] != gen_manifest["experiment_id"]:
        raise ValueError("Konfigurasi/runtime berubah sejak run sebelumnya. Gunakan RUN_NAME baru.")
    write_json(path, gen_manifest)
    return gen_manifest

## 10. Loop generasi K1-K5 dengan resume

In [11]:
_KEEP_EXPERIMENT_ID = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))["experiment_id"]

def freeze_manifest():
    runtime = requests.get(INFERENCE_URL + "/api/version", timeout=15)
    runtime.raise_for_status()
    gen_manifest = {**manifest_18, "protocols": PROTOCOLS, "reference_model": REFERENCE_MODEL,
        "top_k": TOP_K, "models": MODEL_ARTIFACTS, "num_ctx": NUM_CTX,
        "commercial_model": COMMERCIAL_MODEL,
        "prompts_hash": digest([SYSTEM_PROMPT, QUERY_EXPANSION_SYSTEM_PROMPT]),
        "kb_hashes": {n: file_hash(KB_DIR / n) for n in ["chunk_metadata.jsonl", "faiss_index.bin"]},
        "versions": package_versions(), "ollama": runtime.json()}
    gen_manifest["experiment_id"] = _KEEP_EXPERIMENT_ID
    write_json(OUT / "manifest.json", gen_manifest)
    return gen_manifest

In [12]:
from langchain_core.documents import Document as _Doc
 
def restore_docs(snapshot):
    return [_Doc(page_content=d["text"], metadata=d["metadata"]) for d in snapshot["docs"]]
 
def retrieve(question, model_key, vectorstore, bm25):
    start = time.perf_counter()
    expansions = expand_query(question, model_key)
    candidates = hybrid_search(vectorstore, bm25, [question] + expansions)
    scored, ranked = rerank(question, candidates, top_k=TOP_K)
    return {"expansions": expansions, "retrieval_s": time.perf_counter() - start,
            "docs": [{"text": d.page_content, "metadata": d.metadata, "score": float(s)} for d, s in scored],
            "ranked_ids": [evaluation_chunk_id(d.metadata) for d, _ in ranked],
            "candidate_ids": [evaluation_chunk_id(d.metadata) for d in candidates]}
 
def run_one(ex, scenario, protocol, manifest, vectorstore, bm25):
    key = f"{ex['record_id']}__{scenario['model_key']}__{scenario['condition']}__{protocol}"
    path = OUT / "predictions" / (key + ".json")
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        if old.get("experiment_id") == manifest["experiment_id"] and old["status"] == "ok":
            return old
    row = {**scenario, "record_id": ex["record_id"], "protocol": protocol,
           "question": ex["instruction"], "subtask_type": ex.get("subtask_type", ""),
           "experiment_id": manifest["experiment_id"]}
    start = time.perf_counter()
    try:
        snap = {"docs": [], "expansions": [], "retrieval_s": 0.0, "ranked_ids": [], "candidate_ids": []}
        cache_hit = False
        if scenario["use_rag"]:
            if protocol == "controlled":
                cache = OUT / "retrieval_cache" / (ex["record_id"] + ".json")
                if cache.exists():
                    snap = json.loads(cache.read_text(encoding="utf-8"))
                    cache_hit = True
                else:
                    snap = retrieve(ex["instruction"], REFERENCE_MODEL, vectorstore, bm25)
                    snap["experiment_id"] = manifest["experiment_id"]
                    write_json(cache, snap)
            else:
                snap = retrieve(ex["instruction"], scenario["model_key"], vectorstore, bm25)
        docs = restore_docs(snap)
        system_role, repeat_penalty = _system_role(scenario["model_key"])
        messages = build_messages(ex["instruction"], docs, system_role, scenario["use_rag"])
        t = time.perf_counter()
        answer = generate_answer(scenario["model_key"], messages, repeat_penalty, 0, MAX_NEW_TOKENS)
        generation_s = time.perf_counter() - t
        context_text = format_context(docs)
        row.update(status="ok", answer=answer, contexts=[context_text] if docs else [],
                   retrieved_ids=[evaluation_chunk_id(d.metadata) for d in docs],
                   retrieval=snap, context_hash=digest(context_text),
                   retrieval_cache_hit=cache_hit, retrieval_s=snap["retrieval_s"],
                   generation_s=generation_s, total_s=time.perf_counter() - start)
    except Exception as exc:
        row.update(status="error", error=f"{type(exc).__name__}: {exc}", total_s=time.perf_counter() - start)
    write_json(path, row)
    return row
 
def run_experiments():
    manifest = freeze_manifest()
    vectorstore, bm25 = build_indices()
    if "controlled" in PROTOCOLS:
        for ex in test.to_dict("records"):
            cache = OUT / "retrieval_cache" / (ex["record_id"] + ".json")
            if not cache.exists():
                snap = retrieve(ex["instruction"], REFERENCE_MODEL, vectorstore, bm25)
                snap["experiment_id"] = manifest["experiment_id"]
                write_json(cache, snap)
    jobs = [(ex, s, p) for ex in test.to_dict("records") for s in SCENARIOS for p in PROTOCOLS
            if p != "controlled" or s["use_rag"]]
    random.Random(SEED).shuffle(jobs)
    jobs.sort(key=lambda j: j[1]["model_key"])
    records = []
    for i, (ex, s, p) in enumerate(jobs, 1):
        result = run_one(ex, s, p, manifest, vectorstore, bm25)
        records.append(result)
        if i % 20 == 0 or i == len(jobs):
            print(f"{i}/{len(jobs)} {s['model_key']} {s['condition']} {p}: {result['status']}")
    return records
 
RUN_EXPERIMENT = True
if RUN_EXPERIMENT:
    records = run_experiments()
    for model in MODEL_REGISTRY.values():
        requests.post(INFERENCE_URL + "/api/generate",
                      json={"model": model["ollama_name"], "keep_alive": 0}, timeout=60)
    print("Selesai. Lanjut ke notebook 20 untuk Hit@K/Recall@K/MRR + RAGAS.")
else:
    print("RUN_EXPERIMENT=False. Belum ada inferensi dijalankan.")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

20/1246 deepseek-flash K5 controlled: ok
40/1246 deepseek-flash K5 controlled: ok
60/1246 deepseek-flash K5 end_to_end: ok
80/1246 deepseek-flash K5 controlled: ok
100/1246 deepseek-flash K5 end_to_end: ok
120/1246 deepseek-flash K5 end_to_end: ok
140/1246 deepseek-flash K5 end_to_end: ok
160/1246 deepseek-flash K5 end_to_end: ok
180/1246 gemma2-base K2 controlled: ok
200/1246 gemma2-base K2 end_to_end: ok
220/1246 gemma2-base K2 controlled: ok
240/1246 gemma2-base K2 controlled: ok
260/1246 gemma2-base K2 end_to_end: ok
280/1246 gemma2-base K1 end_to_end: ok
300/1246 gemma2-base K2 end_to_end: ok
320/1246 gemma2-base K2 end_to_end: ok
340/1246 gemma2-base K1 end_to_end: ok
360/1246 gemma2-base K2 controlled: ok
380/1246 gemma2-base K1 end_to_end: ok
400/1246 gemma2-base K2 end_to_end: ok
420/1246 gemma2-base K2 controlled: ok
440/1246 gemma2-base K2 controlled: ok
460/1246 gemma2-finetuned K4 controlled: ok
480/1246 gemma2-finetuned K4 controlled: ok
500/1246 gemma2-finetuned K4 end_t